In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import numpy as np
import pandas as pd
import os
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    confusion_matrix
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ==================== SEED SETTING ====================
def set_seed(seed=111):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    import random
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(111)

In [2]:
import torch
import torch.nn as nn
from torchvision import models 
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()
 
    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)
 
 
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
 
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))
 
 
class CBAM(nn.Module):
    def __init__(self, channels, ratio=16):
        super().__init__()
        self.channel_attention = ChannelAttention(channels, ratio)
        self.spatial_attention = SpatialAttention()
 
    def forward(self, x):
        x = x * self.channel_attention(x)
        x = x * self.spatial_attention(x)
        return x
 
 
 
class FeatureCompressor(nn.Module):
    
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(channels, channels // reduction)
        self.relu = nn.ReLU(inplace=True)
 
    def forward(self, x):
        # x: B × C × H × W  →  B × (C/r)
        x = self.avg_pool(x).flatten(1)   # B × C
        x = self.relu(self.fc(x))         # B × C/r
        return x
 
 
class DiseaseDependentAttention(nn.Module):
   
    def __init__(self, channels, reduction=16):
        super().__init__()
        dim = channels // reduction   
 
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(inplace=True),
            nn.Linear(dim, dim),
            nn.Sigmoid(),
        )
 
    def forward(self, g_i, g_j):
        a_c = self.mlp(g_i)          
        attended = g_i * a_c         
        return attended + g_j        
 
 
class CANet(nn.Module):

    def __init__(self, num_dr=5, num_dme=3, reduction=16, pretrained=True):
        super().__init__()
        resnet = models.resnet50(pretrained=pretrained)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])
 
        C = 2048
        dim = C // reduction
 
        self.cbam_dr  = CBAM(C, reduction)
        self.cbam_dme = CBAM(C, reduction)
 
        self.compress_dr  = FeatureCompressor(C, reduction)
        self.compress_dme = FeatureCompressor(C, reduction)
 
       
        self.dda_dr  = DiseaseDependentAttention(C, reduction)
        self.dda_dme = DiseaseDependentAttention(C, reduction)
 
        self.fc_dr_aux  = nn.Linear(dim, num_dr)
        self.fc_dme_aux = nn.Linear(dim, num_dme)
 
        self.fc_dr  = nn.Linear(dim, num_dr)
        self.fc_dme = nn.Linear(dim, num_dme)
 
        self.dropout = nn.Dropout(0.5)
        self._init_weights()
 
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
 
    def forward(self, x):
        feat = self.backbone(x)          
 
        feat_dr  = self.cbam_dr(feat)   
        feat_dme = self.cbam_dme(feat)   
 
        g_dr  = self.compress_dr(feat_dr)    
        g_dme = self.compress_dme(feat_dme)  
 
        out_dr_aux  = self.fc_dr_aux(self.dropout(g_dr))
        out_dme_aux = self.fc_dme_aux(self.dropout(g_dme))
 
       
        g_dr_prime  = self.dda_dr(g_dme, g_dr)   
        g_dme_prime = self.dda_dme(g_dr,  g_dme)  
 
        out_dr  = self.fc_dr(self.dropout(g_dr_prime))
        out_dme = self.fc_dme(self.dropout(g_dme_prime))
 
        return out_dr, out_dme, out_dr_aux, out_dme_aux

In [3]:
class RetinalDataset(Dataset):

    def __init__(self, samples, transform=None):
       
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label_dr, label_dme = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        img_name = os.path.basename(img_path)
        return image, (label_dr, label_dme), img_name


def find_actual_path(base_path, base_name):
   
    nested = os.path.join(base_path, base_name)
    if os.path.isdir(nested):
        return nested
    # Bình thường
    return base_path


def find_annotation_file(folder_path):
  
    try:
        files = os.listdir(folder_path)
    except Exception:
        return None

    for f in files:
        name_lower = f.lower()
        # Kiểm tra bắt đầu bằng "annotation" (có hoặc không có dấu cách/gạch dưới)
        if name_lower.startswith('annotation') and (f.endswith('.xls') or f.endswith('.xlsx')):
            return os.path.join(folder_path, f)
    return None


def load_all_samples(root_dir):
   
    all_samples = []

    base_folders = sorted([
        d for d in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, d)) and d.startswith('Base')
    ])

    print(f'Tìm thấy {len(base_folders)} Base folder: {base_folders}')

    for base_name in base_folders:
        base_path = os.path.join(root_dir, base_name)

        # Tự động phát hiện lồng nhau hay không
        actual_path = find_actual_path(base_path, base_name)

        if not os.path.exists(actual_path):
            print(f'  [WARN] Không tìm thấy: {actual_path}')
            continue

        # Tìm file Annotation (hỗ trợ dấu cách và gạch dưới)
        xls_path = find_annotation_file(actual_path)

        if xls_path is None:
            print(f'  [WARN] Không có file Annotation trong {actual_path}')
            continue

        engine = 'openpyxl' if xls_path.endswith('.xlsx') else 'xlrd'
        try:
            df = pd.read_excel(xls_path, engine=engine)
        except Exception as e:
            print(f'  [WARN] Không đọc được {xls_path}: {e}')
            continue

        col_imgname = df.columns[0]  
        col_dr      = df.columns[2]  
        col_dme     = df.columns[3]  

        count = 0
        for _, row in df.iterrows():
            img_filename = str(row[col_imgname]).strip()
            img_path = os.path.join(actual_path, img_filename)

            if not os.path.exists(img_path):
                continue

            try:
                label_dr  = int(row[col_dr])
                label_dme = int(row[col_dme])
            except (ValueError, TypeError):
                continue

            all_samples.append((img_path, label_dr, label_dme))
            count += 1

        print(f'  {base_name}: {count} ảnh  [{os.path.basename(xls_path)}]')

    print(f'\nTổng: {len(all_samples)} ảnh')
    return all_samples


In [4]:
def get_transforms(train=True, img_size=224):
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
    if train:
        return transforms.Compose([
            transforms.Resize(350),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
            transforms.RandomRotation([-180, 180]),
            transforms.RandomAffine([-180, 180], translate=[0.1, 0.1], scale=[0.7, 1.3]),
            transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
            transforms.ToTensor(),
            normalize
        ])
    else:
        return transforms.Compose([
            transforms.Resize(350),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            normalize,
        ])

In [5]:
class AverageMeter:
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def calculate_metrics(y_true, y_pred, y_prob, num_classes):
    
    acc = accuracy_score(y_true, y_pred)

    # AUC
    if num_classes == 2:
        try:
            auc = roc_auc_score(y_true, y_prob[:, 1])
        except:
            auc = float('nan')
    else:
        try:
            present_classes = np.unique(y_true)
            if len(present_classes) < 2:
                auc = float('nan')
            else:
                y_true_bin = label_binarize(y_true, classes=list(range(num_classes)))
                # Chỉ tính trên class thực sự có trong val
                y_true_bin_present = y_true_bin[:, present_classes]
                y_prob_present = y_prob[:, present_classes]
                auc = roc_auc_score(y_true_bin_present, y_prob_present,
                                    average='macro', multi_class='ovr')
        except Exception as e:
            print(f"[WARN] AUC error: {e}")
            auc = float('nan')
    
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    cm = confusion_matrix(y_true, y_pred)
    if num_classes == 2:
        tn, fp, fn, tp = cm.ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    else:
        binary_true = (y_true > 0).astype(int)
        binary_pred = (y_pred > 0).astype(int)
        cm_bin = confusion_matrix(binary_true, binary_pred)
        if cm_bin.shape == (2, 2):
            tn, fp, fn, tp = cm_bin.ravel()
            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        else:
            sensitivity, specificity = 0.0, 0.0
    
    return {
        'accuracy': acc,
        'auc': auc,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'sensitivity': sensitivity,
        'specificity': specificity
    }

In [6]:

def train_epoch(model, dataloader, criterion, optimizer, device, lambda_aux=0.25):
    
    model.train()
    losses = AverageMeter()
    losses_dr_main = AverageMeter()
    losses_dme_main = AverageMeter()
    losses_dr_aux = AverageMeter()
    losses_dme_aux = AverageMeter()
    
    pbar = tqdm(dataloader, desc='Training')
    for images, (labels_dr, labels_dme), _ in pbar:
        images = images.to(device)
        labels_dr = labels_dr.to(device)
        labels_dme = labels_dme.to(device)
        
        out_dr, out_dme, out_dr_aux, out_dme_aux = model(images)
        
        loss_dr_main = criterion(out_dr, labels_dr)
        loss_dme_main = criterion(out_dme, labels_dme)
        loss_dr_aux = criterion(out_dr_aux, labels_dr)
        loss_dme_aux = criterion(out_dme_aux, labels_dme)
        
        loss = (loss_dr_main + loss_dme_main + 
                lambda_aux * (loss_dr_aux + loss_dme_aux))
        optimizer.zero_grad()
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        losses.update(loss.item(), images.size(0))
        losses_dr_main.update(loss_dr_main.item(), images.size(0))
        losses_dme_main.update(loss_dme_main.item(), images.size(0))
        losses_dr_aux.update(loss_dr_aux.item(), images.size(0))
        losses_dme_aux.update(loss_dme_aux.item(), images.size(0))
        
        pbar.set_postfix({
            'loss': f'{losses.avg:.4f}',
            'dr': f'{losses_dr_main.avg:.4f}',
            'dme': f'{losses_dme_main.avg:.4f}'
        })
    
    return {
        'total': losses.avg,
        'dr_main': losses_dr_main.avg,
        'dme_main': losses_dme_main.avg,
        'dr_aux': losses_dr_aux.avg,
        'dme_aux': losses_dme_aux.avg
    }


def validate_multitask(model, dataloader, criterion, device, lambda_aux=0.25):
    
    model.eval()
    losses = AverageMeter()
    
    all_dr_labels, all_dme_labels = [], []
    all_dr_preds, all_dme_preds = [], []
    all_dr_probs, all_dme_probs = [], []
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validation')
        for images, (labels_dr, labels_dme), _ in pbar:
            images = images.to(device)
            labels_dr_cpu = labels_dr.numpy()
            labels_dme_cpu = labels_dme.numpy()
            labels_dr = labels_dr.to(device)
            labels_dme = labels_dme.to(device)
            
            # Forward pass
            out_dr, out_dme, out_dr_aux, out_dme_aux = model(images)
            
            # Loss
            loss_dr_main = criterion(out_dr, labels_dr)
            loss_dme_main = criterion(out_dme, labels_dme)
            loss_dr_aux = criterion(out_dr_aux, labels_dr)
            loss_dme_aux = criterion(out_dme_aux, labels_dme)
            
            loss = (loss_dr_main + loss_dme_main + 
                    lambda_aux * (loss_dr_aux + loss_dme_aux))
            
            # Predictions (use main outputs only)
            probs_dr = torch.softmax(out_dr, dim=1)
            probs_dme = torch.softmax(out_dme, dim=1)
            preds_dr = torch.argmax(probs_dr, dim=1)
            preds_dme = torch.argmax(probs_dme, dim=1)
            
            # Store results
            all_dr_labels.extend(labels_dr_cpu)
            all_dme_labels.extend(labels_dme_cpu)
            all_dr_preds.extend(preds_dr.cpu().numpy())
            all_dme_preds.extend(preds_dme.cpu().numpy())
            all_dr_probs.extend(probs_dr.cpu().numpy())
            all_dme_probs.extend(probs_dme.cpu().numpy())
            
            losses.update(loss.item(), images.size(0))
            pbar.set_postfix({'loss': f'{losses.avg:.4f}'})
    
    all_dr_labels = np.array(all_dr_labels)
    all_dme_labels = np.array(all_dme_labels)
    all_dr_preds = np.array(all_dr_preds)
    all_dme_preds = np.array(all_dme_preds)
    all_dr_probs = np.array(all_dr_probs)
    all_dme_probs = np.array(all_dme_probs)
    
    metrics_dr = calculate_metrics(all_dr_labels, all_dr_preds, all_dr_probs, num_classes=5)
    metrics_dme = calculate_metrics(all_dme_labels, all_dme_preds, all_dme_probs, num_classes=3)
    
    joint_correct = np.sum(
        (all_dr_preds == all_dr_labels) & (all_dme_preds == all_dme_labels)
    )
    joint_acc = joint_correct / len(all_dr_labels)
    
    return {
        'loss': losses.avg,
        'dr': metrics_dr,
        'dme': metrics_dme,
        'joint_accuracy': joint_acc
    }


In [7]:
def train_canet(
    train_loader,
    val_loader,
    epochs=100,
    lr=0.001,
    weight_decay=1e-4,
    lambda_aux=0.25,
    device='cuda',
    save_dir='models',
    fold=None
):
    fold_tag = f'_fold{fold}' if fold is not None else ''

    model = CANet(num_dr=5, num_dme=3, pretrained=True)
    model = model.to(device)

    print(f'Total parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')
    print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M')

    criterion = nn.CrossEntropyLoss()

    backbone_params = []
    head_params     = []
    for name, param in model.named_parameters():
        if 'backbone' in name:
            backbone_params.append(param)
        else:
            head_params.append(param)

    optimizer = optim.Adam([
        {'params': backbone_params, 'lr': lr * 0.1},
        {'params': head_params,     'lr': lr}
    ], weight_decay=weight_decay)

    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )

    history = {
        'train_loss':    [],
        'val_loss':      [],
        'val_dr_acc':    [],
        'val_dme_acc':   [],
        'val_joint_acc': [],
        'val_dr_auc':    [],
        'val_dme_auc':   []
    }

    best_joint_acc = 0.0
    best_dr_auc    = 0.0
    best_dme_auc   = 0.0

    os.makedirs(save_dir, exist_ok=True)

    for epoch in range(epochs):
        print(f'\n{"="*70}')
        print(f'Epoch {epoch+1}/{epochs}')
        print(f'{"="*70}')

        train_losses = train_epoch(model, train_loader, criterion, optimizer, device, lambda_aux)
        val_results  = validate_multitask(model, val_loader, criterion, device, lambda_aux)

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_losses['total'])
        history['val_loss'].append(val_results['loss'])
        history['val_dr_acc'].append(val_results['dr']['accuracy'])
        history['val_dme_acc'].append(val_results['dme']['accuracy'])
        history['val_joint_acc'].append(val_results['joint_accuracy'])
        history['val_dr_auc'].append(val_results['dr']['auc'])
        history['val_dme_auc'].append(val_results['dme']['auc'])

        print(f'\nLearning Rate: {current_lr:.6f}')
        print(f'Train Loss: {train_losses["total"]:.4f}')
        print(f'  - DR Main: {train_losses["dr_main"]:.4f}  DME Main: {train_losses["dme_main"]:.4f}')
        print(f'  - DR Aux:  {train_losses["dr_aux"]:.4f}   DME Aux:  {train_losses["dme_aux"]:.4f}')
        print(f'Val Loss: {val_results["loss"]:.4f}')
        print(f'DR  Metrics: Acc={val_results["dr"]["accuracy"]:.4f}  AUC={val_results["dr"]["auc"]:.4f}  F1={val_results["dr"]["f1_score"]:.4f}')
        print(f'DME Metrics: Acc={val_results["dme"]["accuracy"]:.4f}  AUC={val_results["dme"]["auc"]:.4f}  F1={val_results["dme"]["f1_score"]:.4f}')
        print(f'Joint Accuracy: {val_results["joint_accuracy"]:.4f}')

        if val_results['joint_accuracy'] > best_joint_acc:
            best_joint_acc = val_results['joint_accuracy']
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'joint_acc': best_joint_acc,
                'dr_metrics': val_results['dr'],
                'dme_metrics': val_results['dme']
            }, os.path.join(save_dir, f'best_joint_acc{fold_tag}.pth'))
            print(f'✓ Saved best joint accuracy model: {best_joint_acc:.4f}')

        if val_results['dr']['auc'] > best_dr_auc:
            best_dr_auc = val_results['dr']['auc']
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'dr_auc': best_dr_auc
            }, os.path.join(save_dir, f'best_dr_auc{fold_tag}.pth'))
            print(f' Saved best DR AUC model: {best_dr_auc:.4f}')

        if val_results['dme']['auc'] > best_dme_auc:
            best_dme_auc = val_results['dme']['auc']
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'dme_auc': best_dme_auc
            }, os.path.join(save_dir, f'best_dme_auc{fold_tag}.pth'))
            print(f'Saved best DME AUC model: {best_dme_auc:.4f}')

    print(f'\n{"="*70}')
    print('Training completed!')
    print(f'Best Joint Accuracy : {best_joint_acc:.4f}')
    print(f'Best DR  AUC        : {best_dr_auc:.4f}')
    print(f'Best DME AUC        : {best_dme_auc:.4f}')
    print(f'{"="*70}')

    return model, history

In [8]:
def main():

    # ==================== CONFIG ====================
    config = {
        'batch_size':   32,
        'epochs':       50,
        'lr':           0.001,
        'weight_decay': 1e-4,
        'lambda_aux':   0.25,
        'img_size':     224,
        'num_workers':  4,
        'n_folds':      10,   # 10-Fold Cross Validation
    }

    # ==================== PATH ====================
    # Chỉ cần sửa ROOT này
    ROOT = '/kaggle/input/datasets/longvu1611/dataset/dataset_zip_pbl4'

    # ==================== DEVICE ====================
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')

    # ==================== LOAD ALL SAMPLES ====================
    print('\nĐang load dataset...')
    all_samples = load_all_samples(ROOT)
    total = len(all_samples)
    print(f'Tổng số ảnh: {total}')
    labels_dr = [s[1] for s in all_samples]  # ← THÊM DÒNG NÀY


    # ==================== 10-FOLD CROSS VALIDATION ====================
    kf = StratifiedKFold(n_splits=config['n_folds'], shuffle=True, random_state=42)
    indices = np.arange(total)

    # Lưu kết quả tổng hợp qua các fold
    cv_results = {
        'fold':         [],
        'dr_acc':       [],
        'dr_auc':       [],
        'dr_f1':        [],
        'dme_acc':      [],
        'dme_auc':      [],
        'dme_f1':       [],
        'joint_acc':    [],
    }

    train_transform = get_transforms(train=True,  img_size=config['img_size'])
    val_transform   = get_transforms(train=False, img_size=config['img_size'])

    for fold, (train_idx, val_idx) in enumerate(kf.split(indices, labels_dr)):  # ← thêm labels_dr
        print(f'\n{"#"*70}')
        print(f'  FOLD {fold+1}/{config["n_folds"]}')
        print(f'  Train: {len(train_idx)} mẫu  |  Val: {len(val_idx)} mẫu')
        print(f'{"#"*70}')

        # Tạo dataset cho fold này
        train_samples = [all_samples[i] for i in train_idx]
        val_samples   = [all_samples[i] for i in val_idx]

        train_dataset = RetinalDataset(train_samples, transform=train_transform)
        val_dataset   = RetinalDataset(val_samples,   transform=val_transform)

        train_loader = DataLoader(
            train_dataset,
            batch_size=config['batch_size'],
            shuffle=True,
            num_workers=0,
            pin_memory=True,
            drop_last=True
        )
        val_loader = DataLoader(
            val_dataset,
            batch_size=config['batch_size'],
            shuffle=False,
            num_workers=0,
            pin_memory=True
        )

        # Train model cho fold này
        model, history = train_canet(
            train_loader,
            val_loader,
            epochs=config['epochs'],
            lr=config['lr'],
            weight_decay=config['weight_decay'],
            lambda_aux=config['lambda_aux'],
            device=device,
            save_dir=f'models/fold_{fold+1}',
            fold=fold+1
        )

        # Lưu kết quả tốt nhất của fold này
        best_epoch = np.argmax(history['val_joint_acc'])
        cv_results['fold'].append(fold+1)
        cv_results['dr_acc'].append(history['val_dr_acc'][best_epoch])
        cv_results['dr_auc'].append(history['val_dr_auc'][best_epoch])
        cv_results['dr_f1'].append(max(history['val_dr_acc']))  # placeholder
        cv_results['dme_acc'].append(history['val_dme_acc'][best_epoch])
        cv_results['dme_auc'].append(history['val_dme_auc'][best_epoch])
        cv_results['dme_f1'].append(max(history['val_dme_acc']))  # placeholder
        cv_results['joint_acc'].append(history['val_joint_acc'][best_epoch])

        print(f'\n[Fold {fold+1}] Best Joint Acc: {cv_results["joint_acc"][-1]:.4f}')
        print(f'[Fold {fold+1}] DR  AUC: {cv_results["dr_auc"][-1]:.4f}')
        print(f'[Fold {fold+1}] DME AUC: {cv_results["dme_auc"][-1]:.4f}')

    # ==================== TỔNG KẾT ====================
    print(f'\n{"="*70}')
    print('10-FOLD CROSS VALIDATION - KẾT QUẢ TỔNG HỢP')
    print(f'{"="*70}')

    df_results = pd.DataFrame(cv_results)
    print(df_results.to_string(index=False))

    print(f'\n--- MEAN ± STD ---')
    for metric in ['dr_acc', 'dr_auc', 'dme_acc', 'dme_auc', 'joint_acc']:
        vals = cv_results[metric]
        print(f'{metric:15s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}')

    # Lưu kết quả ra CSV
    df_results.to_csv('cv_results.csv', index=False)
    print('\nĐã lưu kết quả vào cv_results.csv')


if __name__ == '__main__':
    main()

Using device: cuda

Đang load dataset...
Tìm thấy 12 Base folder: ['Base11', 'Base12', 'Base13', 'Base14', 'Base21', 'Base22', 'Base23', 'Base24', 'Base31', 'Base32', 'Base33', 'Base34']
  Base11: 100 ảnh  [Annotation_Base11.xls]
  Base12: 100 ảnh  [Annotation Base12.xls]
  Base13: 100 ảnh  [Annotation_Base13.xls]
  Base14: 100 ảnh  [Annotation Base14.xls]
  Base21: 100 ảnh  [Annotation Base21.xls]
  Base22: 100 ảnh  [Annotation Base22.xls]
  Base23: 100 ảnh  [Annotation Base23.xls]
  Base24: 100 ảnh  [Annotation Base24.xls]
  Base31: 100 ảnh  [Annotation Base31.xls]
  Base32: 100 ảnh  [Annotation Base32.xls]
  Base33: 100 ảnh  [Annotation Base33.xls]
  Base34: 100 ảnh  [Annotation Base34.xls]

Tổng: 1200 ảnh
Tổng số ảnh: 1200

######################################################################
  FOLD 1/10
  Train: 1080 mẫu  |  Val: 120 mẫu
######################################################################
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" 

100%|██████████| 97.8M/97.8M [00:00<00:00, 191MB/s]


Total parameters: 25.15M
Trainable parameters: 25.15M

Epoch 1/50


Validation: 100%|██████████| 4/4 [00:07<00:00,  1.98s/it, loss=2.2904]



Learning Rate: 0.000098
Train Loss: 2.5109
  - DR Main: 1.3453  DME Main: 0.6471
  - DR Aux:  1.4202   DME Aux:  0.6536
Val Loss: 2.2904
DR  Metrics: Acc=0.4917  AUC=0.6337  F1=0.2404
DME Metrics: Acc=0.7833  AUC=0.7547  F1=0.2928
Joint Accuracy: 0.4500
✓ Saved best joint accuracy model: 0.4500
 Saved best DR AUC model: 0.6337
Saved best DME AUC model: 0.7547

Epoch 2/50


...


Learning Rate: 0.000058
Train Loss: 1.3754
  - DR Main: 0.8008  DME Main: 0.2937
  - DR Aux:  0.8191   DME Aux:  0.3045
Val Loss: 1.8475
DR  Metrics: Acc=0.5750  AUC=0.7491  F1=0.4325
DME Metrics: Acc=0.8583  AUC=0.9334  F1=0.5787
Joint Accuracy: 0.5083

Epoch 49/50


Validation: 100%|██████████| 4/4 [00:03<00:00,  1.14it/s, loss=1.5837]



Learning Rate: 0.000054
Train Loss: 1.3537
  - DR Main: 0.7931  DME Main: 0.2870
  - DR Aux:  0.8052   DME Aux:  0.2894
Val Loss: 1.5837
DR  Metrics: Acc=0.5917  AUC=0.7863  F1=0.4503
DME Metrics: Acc=0.8583  AUC=0.9237  F1=0.4819
Joint Accuracy: 0.5083

Epoch 50/50


Validation: 100%|██████████| 4/4 [00:03<00:00,  1.18it/s, loss=1.5509]


Learning Rate: 0.000051
Train Loss: 1.3638
  - DR Main: 0.7826  DME Main: 0.3027
  - DR Aux:  0.8007   DME Aux:  0.3134
Val Loss: 1.5509
DR  Metrics: Acc=0.6333  AUC=0.7857  F1=0.4751
DME Metrics: Acc=0.9083  AUC=0.9171  F1=0.6275
Joint Accuracy: 0.5917

Training completed!
Best Joint Accuracy : 0.6000
Best DR  AUC        : 0.8213
Best DME AUC        : 0.9506

[Fold 10] Best Joint Acc: 0.6000
[Fold 10] DR  AUC: 0.7897
[Fold 10] DME AUC: 0.9204

10-FOLD CROSS VALIDATION - KẾT QUẢ TỔNG HỢP
 fold   dr_acc   dr_auc    dr_f1  dme_acc  dme_auc   dme_f1  joint_acc
    1 0.666667 0.827601 0.691667 0.891667 0.936327 0.908333   0.616667
    2 0.708333 0.851577 0.708333 0.891667 0.961389 0.925000   0.666667
    3 0.733333 0.851321 0.733333 0.900000 0.937400 0.900000   0.675000
    4 0.733333 0.859625 0.733333 0.900000 0.936073 0.908333   0.666667
    5 0.700000 0.854802 0.700000 0.908333 0.932108 0.933333   0.633333
    6 0.750000 0.849596 0.750000 0.891667 0.942977 0.925000   0.666667
    7 0.6